In [ ]:
import numpy as np
from scipy.sparse.linalg import LinearOperator, svds
from scipy.linalg import sqrtm


# ==========================================
# 1. Matrix-Free Tensor Operations
# ==========================================
def apply_tensor_power(M_base, v, n):
    """
    Computes (M_base ⊗ ... ⊗ M_base) * v without building the full matrix.
    It reshapes the vector into an n-dimensional tensor and applies
    the base matrix along each dimension.
    """
    d = M_base.shape[0]
    # Reshape vector into an n-dimensional grid
    X = v.reshape((d,) * n)
    for i in range(n):
        # Contract M_base with the i-th dimension of X
        X = np.tensordot(M_base, X, axes=([1], [i]))
        # tensordot moves the contracted axis to the front; move it back
        X = np.moveaxis(X, 0, i)
    return X.flatten()


# ==========================================
# 2. Golub-Kahan Bidiagonalization & Quadrature
# ==========================================
def gkb_quadrature(matvec, rmatvec, v, l_steps):
    """
    Runs Golub-Kahan Bidiagonalization (GKB) for l_steps and uses
    Gaussian Quadrature to estimate v^T * sqrt(M^T M) * v.
    """
    N = len(v)
    v_norm = np.linalg.norm(v)
    q = v / v_norm

    omega = np.zeros(l_steps)
    gamma = np.zeros(l_steps)

    p_prev = np.zeros(N)
    gamma_prev = 0.0

    for j in range(l_steps):
        # u-step (building the right singular vectors)
        u_unnorm = matvec(q) - gamma_prev * p_prev
        omega[j] = np.linalg.norm(u_unnorm)

        # Handle lucky breakdown (exact convergence)
        if omega[j] < 1e-12:
            break
        p = u_unnorm / omega[j]

        # v-step (building the left singular vectors)
        if j < l_steps - 1:
            v_unnorm = rmatvec(p) - omega[j] * q
            gamma[j] = np.linalg.norm(v_unnorm)
            if gamma[j] < 1e-12:
                break
            q = v_unnorm / gamma[j]
            gamma_prev = gamma[j]
            p_prev = p

    # Truncate arrays if we hit a breakdown early
    l_actual = j + 1
    omega = omega[:l_actual]
    gamma = gamma[: l_actual - 1]

    # Construct the small upper-bidiagonal matrix B_l
    B_l = np.diag(omega)
    if l_actual > 1:
        B_l += np.diag(gamma, k=1)

    # Form the tridiagonal matrix T_l = B_l^T * B_l
    T_l = B_l.T @ B_l

    # Evaluate the function (matrix square root) on the tiny matrix
    sqrt_T_l = sqrtm(T_l)

    # The Gaussian quadrature estimate is the top-left element scaled by ||v||^2
    quadrature_estimate = (v_norm**2) * sqrt_T_l[0, 0].real
    return quadrature_estimate


# ==========================================
# 3. Main Execution & Testing
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)  # For reproducible random matrices

    # Parameters
    dim = 3
    n = 9
    c1, c2 = 0.5, -0.5
    N_full = dim**n  # 3^5 = 243

    # Generate random 3x3 matrices
    A = np.random.randn(dim, dim)
    B = np.random.randn(dim, dim)

    # Define the Black-Box Operator functions
    def matvec(v):
        return c1 * apply_tensor_power(A, v, n) + c2 * apply_tensor_power(B, v, n)

    def rmatvec(v):
        # M^T uses the transposes of A and B
        return c1 * apply_tensor_power(A.T, v, n) + c2 * apply_tensor_power(B.T, v, n)

    # Create the SciPy LinearOperator interface
    M_op = LinearOperator((N_full, N_full), matvec=matvec, rmatvec=rmatvec)

    print(f"Matrix Dimension: {N_full} x {N_full}\n")

    # ---------------------------------------------------------
    # METHOD 1: Schatten-Infinity Norm (Max Singular Value)
    # ---------------------------------------------------------
    # svds uses implicit restarted Lanczos bidiagonalization
    U, S_max_approx, Vt = svds(M_op, k=1, which="LM")
    schatten_inf = S_max_approx[0]
    print(f"Iterative Schatten-inf Norm: {schatten_inf:.4f}")

    # ---------------------------------------------------------
    # METHOD 2: Schatten-1 Norm (Trace Norm)
    # ---------------------------------------------------------
    # Using Hutchinson's Estimator + GKB Quadrature
    K_samples = 200  # Number of random Rademacher vectors
    l_steps = 30  # Number of Krylov steps per vector

    trace_est = 0.0
    for k in range(K_samples):
        # Generate random vector with entries +1 or -1
        v_rand = np.random.choice([-1.0, 1.0], size=N_full)
        trace_est += gkb_quadrature(matvec, rmatvec, v_rand, l_steps)

    schatten_1 = trace_est / K_samples
    print(f"Iterative Schatten-1 Norm:   {schatten_1:.4f}\n")

    # =========================================================
    # GROUND TRUTH VERIFICATION (Dense Matrix Construction)
    # =========================================================
    print("--- Verifying with exact dense matrix math ---")

    # Recursively build A^{\otimes n}
    A_n = A
    B_n = B
    for _ in range(1, n):
        A_n = np.kron(A_n, A)
        B_n = np.kron(B_n, B)

    M_dense = c1 * A_n + c2 * B_n

    # Compute the full exact SVD
    S_exact = np.linalg.svd(M_dense, compute_uv=False)

    print(f"Exact Schatten-inf Norm:     {S_exact.max():.4f}")
    print(f"Exact Schatten-1 Norm:       {S_exact.sum():.4f}")

Matrix Dimension: 19683 x 19683

Iterative Schatten-inf Norm: 4216.0347
Iterative Schatten-1 Norm:   279578.2810

--- Verifying with exact dense matrix math ---


: 